In [1]:
# Load libraries
library(tidyverse)

# Source the depth profile functions (also loads scenario_classification.R)
source("depth_profile_data.R")

# Load all profile data
profile_data <- load_profile_data()

# Fit proxy model (needed for dynamic mode)
proxy_model <- fit_euphotic_proxy(profile_data$scenario)

# Get full data with scenario-specific depths (35m upwelling, 50m relaxed)
full_data_scenario <- get_full_scenario_data(
  profile_data,
  depth_mode = "scenario",
  scenario_depths = c(upwelling = 35, relaxed = 50),
  trap_lag_months = 0
)

# Or with dynamic (per-date) depths
full_data_dynamic <- get_full_scenario_data(
  profile_data,
  depth_mode = "dynamic",
  proxy_model = proxy_model,
  trap_lag_months = 0
)

# Quick summary
summarize_full_scenario(full_data_scenario)

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.2.0     ✔ readr     2.2.0
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.2     ✔ tibble    3.3.1
✔ lubridate 1.9.5     ✔ tidyr     1.3.2
✔ purrr     1.2.1     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors
Lade nötiges Paket: gsw



Loaded HPLC profiles: 176 dates, depths 0-200m
Loaded cached Niskin profiles: 230 dates, depths 0-200m

=== Date Coverage Summary ===
  Total unique dates:     275
  With HPLC data:         176
  With Niskin data:       230
  With scenario class:    257
  With observed EuZ:      102

Fitting EuZ proxy model (isotherm) on 94 observations...
  Model: EuZ = 20.82 + 0.2683 × Isotherm_21
  R-squared: 0.411 (adj: 0.405)
  RMSE: 10.01 m

Prediction coverage:
  Total dates with Isotherm_21: 229
  With observed EuZ: 94
  With predicted EuZ: 135

=== Building full scenario dataset (mode: scenario) ===
Complete monthly backbone: 255 months
  From 1995-11-01 to 2017-01-01
  With upwelling class: 228 months
  With depth cutoff: 228 months
HPLC: 171 months with data
Niskin: 227 months with data
Zooplankton: 153 months with data
Sediment trap: 160 months with data (lag = 0)

=== Final Data Coverage ===
  Total months (backbone): 255
  With upwelling class: 228
  With depth cutoff: 228
  With TotChlA:

upwelling,n,depth_cutoff,micro,nano,pico,TotChlA,NO3,PON,zoo_gt200,zoo_gt500,PP,export,Temp,Isotherm_21
<chr>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
relaxed,141,50,0.1370079,0.06645553,0.0932409,0.2759719,1.114997,8.028931,0.05229609,0.02798667,933.2341,0.6472141,25.36624,110.29787
upwelling,87,35,0.3249278,0.12916154,0.1026259,0.7082727,2.099959,13.855328,0.07437166,0.04313683,1544.9256,0.8542015,23.09473,59.34483


In [2]:
# Get data
full_data <- get_full_scenario_data(
  profile_data,
  depth_mode = "observed"
)

# Detailed summary with methodology notes
results <- summarize_full_scenario_detailed(full_data, include_unclassified = TRUE)

# Access metadata
View(results$metadata)

# Access raw summary for further processing
results$summary


=== Building full scenario dataset (mode: observed) ===
Complete monthly backbone: 255 months
  From 1995-11-01 to 2017-01-01
  With upwelling class: 228 months
  With depth cutoff: 102 months
HPLC: 79 months with data
Niskin: 102 months with data
Zooplankton: 153 months with data
Sediment trap: 160 months with data (lag = 0)

=== Final Data Coverage ===
  Total months (backbone): 255
  With upwelling class: 228
  With depth cutoff: 102
  With TotChlA: 70
  With phyto size data: 33
  With NO3: 94
  With PP: 97
  With zoo >200µm: 151
  With export flux: 139

╔══════════════════════════════════════════════════════════════════╗
║           CARIACO SCENARIO DATA SUMMARY                          ║
╚══════════════════════════════════════════════════════════════════╝

=== Data Coverage (n observations per variable) ===
 upwelling_group total_dates phyto_size TotChlA NO3 PON PP Temperature
         relaxed         141         24      45  54  55 56          60
    unclassified          27     

variable,units,description,methodology
<chr>,<chr>,<chr>,<chr>
date,Date,Sampling date,From original datasets
depth_cutoff,m,Integration depth for this date,"Fixed (50m), scenario-specific (35/50m), or dynamic (EuZ proxy)"
upwelling,category,Upwelling regime classification,"Based on T at 50m: ≤22°C = upwelling, >22°C = relaxed"
ui,category,Detailed upwelling index,"Based on T at 50m: ≤20°C strong, ≤21°C moderate, ≤22°C weak, >22°C relaxed"
TotChlA_mmolN,mmol N m⁻³,Total Chlorophyll a,"HPLC Tot_Chl_a integrated to depth_cutoff, converted: (mg Chl / depth) × 50 / 12.01 × (16/106)"
micro_mmolN,mmol N m⁻³,Microphytoplankton (>20 µm) Chl a,"Tot_Chl_a partitioned by diagnostic pigment fractions, then converted to mmol N"
nano_mmolN,mmol N m⁻³,Nanophytoplankton (2-20 µm) Chl a,Same as micro_mmolN
pico_mmolN,mmol N m⁻³,Picophytoplankton (<2 µm) Chl a,Same as micro_mmolN
micro_frac,dimensionless,Microphytoplankton fraction of total,From diagnostic pigment ratios


upwelling_group,n_dates,depth_cutoff_mean,n_phyto,micro_mean,micro_sd,nano_mean,nano_sd,pico_mean,pico_sd,⋯,zoo_gt500_sd,n_PP,PP_mean,PP_sd,n_export,export_mean,export_sd,Temp_mean,Temp_sd,Isotherm_21_mean
<chr>,<int>,<dbl>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<int>,<dbl>,<dbl>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
relaxed,141,51.35063,24,0.1403349,0.3269070,0.06679423,0.02598453,0.09822235,0.03243180,⋯,0.01876230,56,894.0087,561.2857,86,0.6472141,0.3307737,25.20929,1.2035897,110.29787
unclassified,27,57.00000,0,NaN,NA,NaN,NA,NaN,NA,⋯,0.01838313,1,365.8462,NA,6,0.4898344,0.4327909,26.64418,NA,NaN
upwelling,87,35.20409,9,0.4046525,0.7964615,0.07941531,0.02497322,0.08552875,0.04279666,⋯,0.02498598,40,1637.2083,840.8589,47,0.8542015,0.5389891,22.98130,0.7603573,59.34483


In [7]:
profile_data$niskin

date,depth,NO3_merged,Chlorophyll,PrimaryProductivity,PN_ug_L,Temperature
<date>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1995-11-08,0,0.1800000,0.09407620,NA,NA,27.49000
1995-11-08,1,0.1800000,0.09407620,NA,NA,27.49000
1995-11-08,2,0.1793403,0.09304273,NA,NA,27.49054
1995-11-08,3,0.1769706,0.09034286,NA,NA,27.49198
1995-11-08,4,0.1747408,0.08598268,NA,NA,27.49434
1995-11-08,5,0.1728265,0.07972029,NA,NA,27.49752
1995-11-08,6,0.1709142,0.07564979,NA,NA,27.49944
1995-11-08,7,0.1672057,0.09397355,NA,NA,27.49941
1995-11-08,8,0.1655658,0.09432946,NA,NA,27.49824


In [3]:
# Load data
profile_data <- load_profile_data()

# Option 1: Original isotherm-only model (backward compatible)
proxy_model <- fit_euphotic_proxy(profile_data$scenario)

# Option 2: Improved isotherm + chlorophyll model
proxy_model_improved <- fit_euphotic_proxy(
  profile_data$scenario,
  niskin_profiles = profile_data$niskin,
  model_type = "isotherm_chl"
)

# Option 3: Compare both models
model_comparison <- compare_euphotic_proxy_models(
  profile_data$scenario,
  profile_data$niskin
)

# Use improved model for dynamic depth integration
full_data_dynamic <- get_full_scenario_data(
  profile_data,
  depth_mode = "dynamic",
  proxy_model = proxy_model_improved
)

Loaded HPLC profiles: 176 dates, depths 0-200m
Loaded cached Niskin profiles: 230 dates, depths 0-200m

=== Date Coverage Summary ===
  Total unique dates:     275
  With HPLC data:         176
  With Niskin data:       230
  With scenario class:    257
  With observed EuZ:      102

Fitting EuZ proxy model (isotherm) on 94 observations...
  Model: EuZ = 20.82 + 0.2683 × Isotherm_21
  R-squared: 0.411 (adj: 0.405)
  RMSE: 10.01 m

Prediction coverage:
  Total dates with Isotherm_21: 229
  With observed EuZ: 94
  With predicted EuZ: 135
Chlorophyll integration (0-100m):
  Dates with sufficient Chl data: 229

Fitting EuZ proxy model (isotherm_chl) on 94 observations...
  Model: EuZ = 81.55 + 0.0983 × Isotherm_21 + -30.4234 × log10(Chl_int)
  Coefficient signs: Isotherm ✓ (expect +), log(Chl) ✓ (expect -)
  R-squared: 0.717 (adj: 0.710)
  RMSE: 6.94 m

Prediction coverage:
  Total dates with Isotherm_21: 229
  With observed EuZ: 94
  With predicted EuZ (iso+chl): 104
  No prediction (miss